<a href="https://colab.research.google.com/github/diaoumardia2001-beep/DI-Bootcamp-May/blob/main/Exercises_XP_RAG_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercises XP: RAG with LangChain (Student)

## 0) Setup


In [ ]:
!pip -q install -U datasets transformers sentence-transformers faiss-cpu langchain langchain-core langchain-community langchain-text-splitters langchain-huggingface

In [ ]:
from typing import List

from datasets import load_dataset
from transformers import pipeline

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA


## 1) Load dataset and convert to Documents


In [ ]:
dataset_name = "m-ric/huggingface_doc"
split = "train[:200]"
text_column = "text"
source_column = "source"

ds = load_dataset(dataset_name, split=split)

print("Columns:", ds.column_names)
print("Example row:", ds[0])

## 2) Split into chunks


In [ ]:
documents: List[Document] = []
for i, row in enumerate(ds):
    documents.append(
        Document(
            page_content=row[text_column],
            metadata={
                "source": row[source_column],
                "doc_id": i
            }
        )
    )

print("Documents:", len(documents))
print("Example:", documents[0].metadata)
print(documents[0].page_content[:350])


In [ ]:
def chunk_documents(documents, chunk_size, chunk_overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    return splitter.split_documents(documents)

chunks_v1 = chunk_documents(documents, chunk_size=500, chunk_overlap=50)
print(f"[chunk_size=500, overlap=50] -> {len(chunks_v1)} chunks")
print(chunks_v1[0].page_content[:200])

chunks_v2 = chunk_documents(documents, chunk_size=1000, chunk_overlap=100)
print(f"\n[chunk_size=1000, overlap=100] -> {len(chunks_v2)} chunks")
print(chunks_v2[0].page_content[:200])

## 3) Vector store + retriever (FAISS)


In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = FAISS.from_documents(chunks_v1, embedding_model)

retriever_k4 = vector_store.as_retriever(search_kwargs={"k": 4})
retriever_k2 = vector_store.as_retriever(search_kwargs={"k": 2})
retriever_k6 = vector_store.as_retriever(search_kwargs={"k": 6})

Retriever ready


## 4) Build the RAG chain


In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA  # on latest stack

llm_id = "google/flan-t5-small"
hf = pipeline(
    ... # To-Do: fill in the pipeline appropriately, remember that you are using a text2text-generation task
)

llm = HuggingFacePipeline(pipeline=hf)

qa = RetrievalQA.from_chain_type(
    ... # To-Do: fill in the llm and retriever with the following parameters: llm, retriever, chain_type
)

print("RAG chain ready")


In [ ]:
test_question = "How do I use the Trainer API in Hugging Face?"

for k, retriever in [(2, retriever_k2), (4, retriever_k4), (6, retriever_k6)]:
    print(f"\n=== k={k} ===")
    results = retriever.invoke(test_question)
    for j, doc in enumerate(results):
        print(f"--- Chunk {j+1} (source: {doc.metadata['source']}) ---")
        print(doc.page_content[:200], "...\n")

## 5) Demo: RAG vs no-RAG


In [ ]:
generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_length=256
)

llm = HuggingFacePipeline(pipeline=generator)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever_k4,
    return_source_documents=True
)

questions = [
    "How do I use the Trainer API in Hugging Face?",
    "What is a tokenizer?",
    "How do I push a model to the Hugging Face Hub?"
]

for q in questions:
    result = qa_chain.invoke({"query": q})
    print(f"\nQ: {q}")
    print(f"A: {result['result']}")
    print("Sources:")
    for doc in result["source_documents"]:
        print(" -", doc.metadata["source"])